The goal of the notebook is to create an output path and a meta-data list of the form

```
metadata_list = [
    {  # Can create an artibtrary number of groups
        "group_name": "Unstimulated",
        "org_shortname": "cionic",
        "group_color": "steelblue",
        "recordings": [  # Add new recordings to list as needed
            {
                "study_shortname": "Parkinsons",
                "collection_num": 55,
                "label": "unstimulated_walk",
            },
            {
                "study_shortname": "Parkinsons",
                "collection_num": 59,
                "label": "unstimulated_walk",
            },
        ],
    },
    {
        "group_name": "Stimulated",
        "org_shortname": "cionic",
        "group_color": "sandybrown",
        "recordings": [  # Add new recordings to list as needed
            {
                "study_shortname": "Parkinsons",
                "collection_num": 59,
                "label": "stim_walk",
            },
            {
                "study_shortname": "Parkinsons",
                "collection_num": 86,
                "label": "stim_walk",
            },
        ],
    },
]
```


```
colors = [
    "firebrick",
    "sandybrown",
    "steelblue",
    "olivedrab",
    "mediumpurple",
    "slategray",
    "peru",
    "cadetblue",
]
```

The gui should help the user walk-through creation of the metadata list data structure and be inspired by the contents of runner_new.iypnb. As the meta-data list is being created, the resulting json should be rendered below the widget input interface with widgets.Textarea. The workflow is as follows.

- [] create output output path
  - generate optional path comprised of 2 random words while providing input text user to create a specfic 
- [] specify the org shortname
- [] specify number of groups in the metadata list
- [] loop through the number of groups. For each group
    - ask for a group-name
    - assign a group color based on the colors list
    - add recording with a drop-down for study short name, text entry for collection number, and text entry for label
    - after adding one recording, ask whether to add another recording or advanced to next group

please write the code

In [ ]:
import json
import os
import random

import ipywidgets as widgets
from IPython.display import clear_output, display
from ipywidgets import Layout

from cionic import api

# Color palette
COLORS = [
    "firebrick",
    "sandybrown",
    "steelblue",
    "olivedrab",
    "mediumpurple",
    "slategray",
    "peru",
    "cadetblue",
]

# Word lists for random path generation
ADJECTIVES = [
    "swift",
    "bright",
    "clever",
    "smooth",
    "quiet",
    "bold",
    "wise",
    "calm",
    "keen",
    "quick",
    "sharp",
    "clear",
    "deep",
    "strong",
    "light",
    "pure",
]
NOUNS = [
    "falcon",
    "river",
    "mountain",
    "forest",
    "ocean",
    "star",
    "thunder",
    "wind",
    "crystal",
    "flame",
    "shadow",
    "dream",
    "storm",
    "dawn",
    "sunset",
    "moon",
]

BUTTON_WIDTH = "300px"


class MetadataListCreator:
    """Interactive widget interface for creating metadata lists for analysis."""

    def __init__(self, tokenpath: str = '../../token.json'):
        self.tokenpath = os.path.abspath(tokenpath)

        # Data storage
        self.organizations = []
        self.studies = {}
        self.metadata_list = []
        self.current_group = {}

        self.color_index = 0

        # State tracking
        self.num_groups = 0
        self.current_group_index = 0
        self.output_path = ""
        self.org_shortname = ""

        self._create_widgets()
        self._load_organizations()
        self._show_output_path_step()

    def _create_widgets(self):
        """Create all UI widgets."""

        # Step container - use Output widget for context manager support
        self.step_output = widgets.Output(layout=Layout(width='600px', padding='10px'))

        # JSON output display
        self.json_output = widgets.Textarea(
            value='[]',
            rows=15,
            description="Metadata JSON:",
            layout=Layout(width='800px', height='400px'),
            disabled=True,
        )

        # Main container
        self.main_container = widgets.VBox(
            [
                widgets.HTML("<h2>Metadata List Creator</h2>"),
                self.step_output,
                widgets.HTML("<h3>Generated Metadata:</h3>"),
                self.json_output,
            ]
        )

    def _load_organizations(self):
        """Load available organizations from API."""
        try:
            auth_data = api.auth(tokenpath=self.tokenpath)
            self.organizations = [org['shortname'] for org in auth_data]
        except Exception as e:
            print(f"Failed to load organizations: {e}")
            self.organizations = []

    def _generate_random_path(self) -> str:
        """Generate a random path name using adjective + noun."""
        adj = random.choice(ADJECTIVES)
        noun = random.choice(NOUNS)
        return f"{adj}_{noun}"

    def _show_output_path_step(self):
        """Show the output path creation step."""
        with self.step_output:
            clear_output(wait=True)

            random_path = self._generate_random_path()

            path_input = widgets.Text(
                value=random_path,
                description="Output Path:",
                placeholder="Enter custom path or use generated one",
                layout=Layout(width='400px'),
            )

            next_button = widgets.Button(
                description="Next: Select Organization",
                button_style='primary',
                layout=Layout(width=BUTTON_WIDTH),
            )

            def on_next_clicked(b):
                self.output_path = path_input.value.strip()
                if self.output_path:
                    self._show_org_selection_step()
                else:
                    print("Please enter an output path")

            next_button.on_click(on_next_clicked)

            display(
                widgets.VBox(
                    [
                        widgets.HTML("<h3>Step 1: Create Output Path</h3>"),
                        widgets.HTML(
                            "<p>Specify the output path for your analysis:</p>"
                        ),
                        path_input,
                        widgets.HTML(
                            f"<p><small>Random suggestion: {random_path}</small></p>"
                        ),
                        next_button,
                    ]
                )
            )

    def _show_org_selection_step(self):
        """Show organization selection step."""
        with self.step_output:
            clear_output(wait=True)

            org_select = widgets.Dropdown(
                options=self.organizations,
                description="Organization:",
                layout=Layout(width='300px'),
            )

            next_button = widgets.Button(
                description="Next: Number of Groups",
                button_style='primary',
                layout=Layout(width=BUTTON_WIDTH),
            )

            back_button = widgets.Button(
                description="Back",
                button_style='warning',
                layout=Layout(width=BUTTON_WIDTH),
            )

            def on_next_clicked(b):
                if org_select.value:
                    self.org_shortname = org_select.value
                    self._load_studies_for_org()
                    self._show_group_count_step()
                else:
                    print("Please select an organization")

            def on_back_clicked(b):
                self._show_output_path_step()

            next_button.on_click(on_next_clicked)
            back_button.on_click(on_back_clicked)

            display(
                widgets.VBox(
                    [
                        widgets.HTML("<h3>Step 2: Select Organization</h3>"),
                        widgets.HTML(
                            f"<p>Output path: <code>{self.output_path}</code></p>"
                        ),
                        org_select,
                        widgets.HBox([back_button, next_button]),
                    ]
                )
            )

    def _load_studies_for_org(self):
        """Load studies for the selected organization."""
        try:
            studies_data = api.get_cionic(f'{self.org_shortname}/studies')
            self.studies[self.org_shortname] = [s['shortname'] for s in studies_data]
        except Exception as e:
            print(f"Failed to load studies: {e}")
            self.studies[self.org_shortname] = []

    def _show_group_count_step(self):
        """Show step to specify number of groups."""
        with self.step_output:
            clear_output(wait=True)

            count_input = widgets.IntText(
                value=2,
                description="# Groups:",
                min=1,
                max=len(COLORS),
                layout=Layout(width='200px'),
            )

            next_button = widgets.Button(
                description="Next: Create Groups",
                button_style='primary',
                layout=Layout(width=BUTTON_WIDTH),
            )

            back_button = widgets.Button(
                description="Back",
                button_style='warning',
                layout=Layout(width=BUTTON_WIDTH),
            )

            def on_next_clicked(b):
                if count_input.value > 0:
                    self.num_groups = count_input.value
                    self.current_group_index = 0
                    self.color_index = 0
                    self._show_group_creation_step()
                else:
                    print("Please enter a valid number of groups")

            def on_back_clicked(b):
                self._show_org_selection_step()

            next_button.on_click(on_next_clicked)
            back_button.on_click(on_back_clicked)

            display(
                widgets.VBox(
                    [
                        widgets.HTML("<h3>Step 3: Specify Number of Groups</h3>"),
                        widgets.HTML(
                            f"<p>Organization: <code>{self.org_shortname}</code></p>"
                        ),
                        widgets.HTML(
                            f"<p>Output path: <code>{self.output_path}</code></p>"
                        ),
                        count_input,
                        widgets.HTML(
                            f"<p><small>Maximum {len(COLORS)} groups"
                            f"(limited by color palette)</small></p>"
                        ),
                        widgets.HBox([back_button, next_button]),
                    ]
                )
            )

    def _show_group_creation_step(self):
        """Show group creation step for current group."""
        with self.step_output:
            clear_output(wait=True)

            # Initialize current group if needed
            if not hasattr(self, 'current_group') or not self.current_group:
                self.current_group = {
                    "group_name": "",
                    "org_shortname": self.org_shortname,
                    "group_color": COLORS[self.color_index % len(COLORS)],
                    "recordings": [],
                }

            group_name_input = widgets.Text(
                value=self.current_group.get("group_name", ""),
                description="Group Name:",
                layout=Layout(width='300px'),
            )

            color_display = widgets.HTML(
                f"<p>Assigned Color: <span style='color: "
                f"{self.current_group['group_color']}; font-weight: bold;'>"
                f"{self.current_group['group_color']}</span></p>"
            )

            next_button = widgets.Button(
                description="Next: Add Recording",
                button_style='primary',
                layout=Layout(width=BUTTON_WIDTH),
            )

            back_button = widgets.Button(
                description="Back",
                button_style='warning',
                layout=Layout(width=BUTTON_WIDTH),
            )

            def on_next_clicked(b):
                if group_name_input.value.strip():
                    self.current_group["group_name"] = group_name_input.value.strip()
                    self._show_recording_creation_step()
                else:
                    print("Please enter a group name")

            def on_back_clicked(b):
                if self.current_group_index == 0:
                    self._show_group_count_step()
                else:
                    self.current_group_index -= 1
                    self.color_index -= 1
                    self.current_group = self.metadata_list.pop()
                    self._show_group_creation_step()

            next_button.on_click(on_next_clicked)
            back_button.on_click(on_back_clicked)

            display(
                widgets.VBox(
                    [
                        widgets.HTML(
                            f"<h3>Step 4: Create Group {self.current_group_index + 1} "
                            f"of {self.num_groups}</h3>"
                        ),
                        widgets.HTML(
                            f"<p>Organization: <code>{self.org_shortname}</code></p>"
                        ),
                        group_name_input,
                        color_display,
                        widgets.HBox([back_button, next_button]),
                    ]
                )
            )

    def _show_recording_creation_step(self):
        """Show recording creation step."""
        with self.step_output:
            clear_output(wait=True)

            study_select = widgets.Dropdown(
                options=self.studies.get(self.org_shortname, []),
                description="Study:",
                layout=Layout(width='300px'),
            )

            collection_input = widgets.IntText(
                description="Collection #:", layout=Layout(width='200px')
            )

            label_input = widgets.Text(
                description="Label:",
                placeholder="e.g., stim_walk",
                layout=Layout(width='300px'),
            )

            add_recording_button = widgets.Button(
                description="Add Recording",
                button_style='success',
                layout=Layout(width=BUTTON_WIDTH),
            )

            # Show current recordings
            recordings_display = widgets.HTML(self._format_current_recordings())

            # Navigation buttons
            if len(self.current_group["recordings"]) > 0:
                if self.current_group_index < self.num_groups - 1:
                    next_action = "Next Group"
                else:
                    next_action = "Finish"

                next_group_button = widgets.Button(
                    description=next_action,
                    button_style='primary',
                    layout=Layout(width=BUTTON_WIDTH),
                )

                def on_next_group_clicked(b):
                    self._finalize_current_group()

            else:
                next_group_button = widgets.HTML(
                    "<p><em>Add at least one recording to continue</em></p>"
                )

            back_button = widgets.Button(
                description="Back to Group Settings",
                button_style='warning',
                layout=Layout(width=BUTTON_WIDTH),
            )

            def on_add_recording_clicked(b):
                if (
                    study_select.value
                    and collection_input.value
                    and label_input.value.strip()
                ):
                    recording = {
                        "study_shortname": study_select.value,
                        "collection_num": collection_input.value,
                        "label": label_input.value.strip(),
                    }
                    self.current_group["recordings"].append(recording)

                    # Clear inputs
                    collection_input.value = 0
                    label_input.value = ""

                    # Refresh display
                    self._show_recording_creation_step()
                    self._update_json_output()
                else:
                    print("Please fill in all recording fields")

            def on_back_clicked(b):
                self._show_group_creation_step()

            add_recording_button.on_click(on_add_recording_clicked)
            back_button.on_click(on_back_clicked)

            if hasattr(next_group_button, 'on_click'):
                next_group_button.on_click(on_next_group_clicked)

            buttons = [back_button, add_recording_button]
            if hasattr(next_group_button, 'on_click'):
                buttons.append(next_group_button)

            display(
                widgets.VBox(
                    [
                        widgets.HTML(
                            f"<h3>Add Recordings to Group: "
                            f"{self.current_group['group_name']}</h3>"
                        ),
                        study_select,
                        collection_input,
                        label_input,
                        recordings_display,
                        widgets.HBox(buttons),
                    ]
                )
            )

    def _format_current_recordings(self) -> str:
        """Format current recordings for display."""
        if not self.current_group["recordings"]:
            return "<p><em>No recordings added yet</em></p>"

        html = "<h4>Current Recordings:</h4><ul>"
        for rec in self.current_group["recordings"]:
            html += f"<li><strong>{rec['study_shortname']}</strong> - Collection "
            f"{rec['collection_num']} ({rec['label']})</li>"
        html += "</ul>"
        return html

    def _finalize_current_group(self):
        """Finalize current group and move to next or finish."""
        # Add current group to metadata list
        self.metadata_list.append(self.current_group.copy())
        self._update_json_output()

        # Move to next group or finish
        self.current_group_index += 1
        self.color_index += 1

        if self.current_group_index < self.num_groups:
            # Start next group
            self.current_group = {
                "group_name": "",
                "org_shortname": self.org_shortname,
                "group_color": COLORS[self.color_index % len(COLORS)],
                "recordings": [],
            }
            self._show_group_creation_step()
        else:
            # Finished all groups
            self._show_completion_step()

    def _show_completion_step(self):
        """Show completion step with final results."""
        with self.step_output:
            clear_output(wait=True)

            # Create save button
            save_button = widgets.Button(
                description="Save to File",
                button_style='success',
                layout=Layout(width=BUTTON_WIDTH),
            )

            restart_button = widgets.Button(
                description="Create New List",
                button_style='info',
                layout=Layout(width=BUTTON_WIDTH),
            )

            def on_save_clicked(b):
                filename = f"metadata_{self.output_path}.json"
                try:
                    os.makedirs("metadata", exist_ok=True)
                    filepath = f"metadata/{filename}"
                    with open(filepath, 'w') as f:
                        json.dump(self.metadata_list, f, indent=2)
                    print(f"Metadata saved to: {filepath}")
                except Exception as e:
                    print(f"Error saving file: {e}")

            def on_restart_clicked(b):
                self.__init__(self.tokenpath)
                self.display()

            save_button.on_click(on_save_clicked)
            restart_button.on_click(on_restart_clicked)

            summary = f"""
            <h3>Metadata List Complete!</h3>
            <p><strong>Output Path:</strong> {self.output_path}</p>
            <p><strong>Organization:</strong> {self.org_shortname}</p>
            <p><strong>Groups Created:</strong> {len(self.metadata_list)}</p>
            <p><strong>Total Recordings:</strong> {
                sum(len(group['recordings']) for group in self.metadata_list)
            }</p>
            """

            display(
                widgets.VBox(
                    [widgets.HTML(summary), widgets.HBox([save_button, restart_button])]
                )
            )

    def _update_json_output(self):
        """Update the JSON output display."""
        # Create a temporary list including current group if it has recordings
        temp_list = self.metadata_list.copy()
        if (
            hasattr(self, 'current_group')
            and self.current_group
            and self.current_group.get("recordings")
        ):
            temp_list.append(self.current_group)

        self.json_output.value = json.dumps(temp_list, indent=2)

    def display(self):
        """Display the metadata creator interface."""
        display(self.main_container)


# Create and display the metadata creator
metadata_creator = MetadataListCreator()
metadata_creator.display()

In [ ]:
# from pprint import pprint

# pprint(metadata_creator.metadata_list)